# Daily morning board

Run top → bottom after the CLI morning loop (`grade_projections`, `log_projections`, `odds_board`, `poll_odds open`).
Pure slate/board generator — for CLV, PnL, edge-bin, and other results analysis,
see `production/notebooks/results_dashboard.ipynb` instead.

1. **Yesterday** — graded `expected_K` vs actuals  
2. **Today** — preferred SP projections  
3. **Edges** — model × live DK/FD (`recommendations.parquet`)

Batting orders = RotoGrinders; dual SP rows collapse to **preferred** (MLB on disagreement).

In [ ]:
from __future__ import annotations

import pickle
import subprocess
import sys
import tempfile
from datetime import date, datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import polars as pl
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "Python").exists() and (candidate / "production").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(f"Cannot find src/Python from cwd={Path.cwd()}")

sys.path.insert(0, str(ROOT / "src"))

from Python.kpi_policy import load_kpi_policy
from Python.notebook_analysis_utils import (
    has_over_clv_red_flag,
    keep_best_available_lines,
    side_clv_roi_health,
)

WORKER = ROOT / "production" / "notebooks" / "_notebook_score.py"
ALLOW_STALE = True
# Overnight RG often still shows yesterday after ET midnight. If the board
# scorer says cards match YESTERDAY, set: SLATE_DATE = YESTERDAY
# Leave None for "today ET" once RG has flipped.
SLATE_DATE: date | None = None
QUIET_WARNINGS = True
ET = ZoneInfo("America/New_York")
TODAY = datetime.now(ET).date()
YESTERDAY = TODAY - timedelta(days=1)

GRADED_PATH = ROOT / "artifacts" / "projection_log" / "graded.parquet"
REC_PATH = ROOT / "artifacts" / "odds_log" / "recommendations.parquet"
LEDGER_PATH = ROOT / "artifacts" / "odds_log" / "ledger.parquet"


def _attach_pitcher_team(df: pl.DataFrame) -> pl.DataFrame:
    if "is_home" not in df.columns or "away_team" not in df.columns:
        return df
    return df.with_columns(
        pl.when(pl.col("is_home"))
        .then(pl.col("home_team"))
        .otherwise(pl.col("away_team"))
        .alias("pitcher_team"),
    )


def _round_display_pdf(df: pl.DataFrame):
    return df.to_pandas().round(3)


def show_scrollable(df: pl.DataFrame, height: int = 420):
    pdf = _round_display_pdf(df)
    table = pdf.to_html(index=False, classes="proj-board", na_rep="—")
    display(
        HTML(
            f"""
<style>
  .proj-scroll {{ max-height: {height}px; overflow: auto; border: 1px solid #4443; border-radius: 6px; }}
  .proj-scroll table.proj-board {{ border-collapse: collapse; width: max-content; min-width: 100%; font-size: 13px; }}
  .proj-scroll thead th {{ position: sticky; top: 0; background: var(--jp-layout-color1, #1e1e1e); z-index: 1; text-align: left; padding: 6px 10px; white-space: nowrap; }}
  .proj-scroll tbody td {{ text-align: left; padding: 4px 10px; white-space: nowrap; }}
</style>
<div class="proj-scroll">{table}</div>
"""
        )
    )


def show_table(df: pl.DataFrame, max_rows: int = 30, height: int = 420):
    pdf = _round_display_pdf(df)
    if len(pdf) <= max_rows:
        display(pdf)
    else:
        show_scrollable(df, height=height)


def _summary(df: pl.DataFrame, label: str) -> dict:
    if df.is_empty():
        return {"label": label, "n": 0}
    ek = df["expected_K"].to_numpy()
    ak = df["actual_K"].to_numpy()
    resid = ek - ak
    return {
        "label": label,
        "n": int(df.height),
        "mae_K": round(float(np.mean(np.abs(resid))), 3),
        "bias_K": round(float(np.mean(resid)), 3),
        "rmse_K": round(float(np.sqrt(np.mean(resid**2))), 3),
        "corr": round(float(np.corrcoef(ek, ak)[0, 1]), 3) if len(ek) > 1 else None,
        "within_1K": round(float(np.mean(np.abs(resid) <= 1.0)), 3),
    }


print("repo:", ROOT)
print("today ET:", TODAY, "| yesterday ET:", YESTERDAY)

## 1. Score today's slate

Needed for today's projections below. Quiet worker subprocess.

In [ ]:
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as tmp:
    out_path = Path(tmp.name)

cmd = [sys.executable, str(WORKER), str(out_path)]
if ALLOW_STALE:
    cmd.append("--allow-stale")
if SLATE_DATE is not None:
    cmd.extend(["--date", SLATE_DATE.isoformat()])
if QUIET_WARNINGS:
    cmd.append("--quiet")

proc = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(
        "Board scorer failed.\n"
        f"cmd: {' '.join(cmd)}\nstdout:\n{proc.stdout}\nstderr:\n{proc.stderr}"
    )

payload = pickle.loads(out_path.read_bytes())
out_path.unlink(missing_ok=True)

board: pl.DataFrame = payload["board"]
preferred: pl.DataFrame = payload["preferred"]
build_meta = payload["build_meta"]
report = payload["report"]
slate_date = str(build_meta.get("slate_date"))

print(
    "slate:", slate_date,
    "| preferred:", preferred.height,
    "| mean xK:", round(float(report["mean_expected_K"]), 3),
)
print("rolling_max:", build_meta.get("rolling_max_date"), "stale_days:", build_meta.get("stale_days"))

slate: 2026-08-21 | preferred: 30 | mean xK: 4.955
rolling_max: 2026-08-20 stale_days: 1


## 2. Yesterday — projections vs actuals

From `graded.parquet`. Excludes pregame OOS and abbreviated outings when those flags exist.

`residual_K = expected_K − actual_K` (positive = over-projected).

In [ ]:
matched = pl.DataFrame()
prev_day = pl.DataFrame()
if not GRADED_PATH.exists():
    print(f"Missing {GRADED_PATH}. Run grade_projections after Level 1 has actuals.")
else:
    graded = pl.read_parquet(GRADED_PATH).with_columns(pl.col("game_date").cast(pl.Date))
    if "is_preferred" in graded.columns:
        graded = graded.filter(pl.col("is_preferred"))
    if "grade_preferred_only" in graded.columns:
        pref_g = graded.filter(pl.col("grade_preferred_only") == True)  # noqa: E712
        if pref_g.height:
            graded = pref_g
    if "is_out_of_support" in graded.columns:
        graded = graded.filter(~pl.col("is_out_of_support").fill_null(False))
    if "is_abbreviated_outing" in graded.columns:
        graded = graded.filter(~pl.col("is_abbreviated_outing").fill_null(False))
    matched = graded.filter(pl.col("has_actual") & pl.col("actual_K").is_not_null())
    prev_day = matched.filter(pl.col("game_date") == YESTERDAY)

    display(pl.DataFrame([_summary(prev_day, f"yesterday_{YESTERDAY}")]).to_pandas())
    if prev_day.is_empty():
        print(f"No graded actuals for {YESTERDAY} yet.")
    else:
        prev = prev_day
        if "pitcher_team" not in prev.columns:
            prev = _attach_pitcher_team(prev)

        # Add prior-day posted line and side recommendations from ledger rows.
        if LEDGER_PATH.exists() and {"game_date", "player_name"}.issubset(prev.columns):
            ledger_day = (
                pl.read_parquet(LEDGER_PATH)
                .with_columns(pl.col("game_date").cast(pl.Date))
                .filter(pl.col("game_date") == YESTERDAY)
            )
            if {"game_date", "player_name", "line", "side"}.issubset(ledger_day.columns):
                rec_flags = (
                    ledger_day
                    .with_columns(pl.col("side").cast(pl.Utf8).str.to_lowercase().alias("_side"))
                    .group_by(["game_date", "player_name"])
                    .agg(
                        [
                            pl.col("line").drop_nulls().first().alias("line"),
                            pl.when((pl.col("_side") == "over").any())
                            .then(pl.lit("over"))
                            .when((pl.col("_side") == "under").any())
                            .then(pl.lit("under"))
                            .otherwise(pl.lit(None, dtype=pl.Utf8))
                            .alias("lean"),
                        ]
                    )
                )
                prev = prev.join(rec_flags, on=["game_date", "player_name"], how="left")

        gcols = [
            c
            for c in (
                "player_name",
                "pitcher_team",
                "away_team",
                "home_team",
                "line",
                "lean",
                "expected_K",
                "actual_K",
                "residual_K",
                "projected_tbf",
                "actual_PA",
            )
            if c in prev.columns
        ]
        prev = (
            prev.select(gcols)
            .with_columns(
                [
                    pl.col(c).round(2)
                    for c in ("line", "expected_K", "residual_K", "projected_tbf")
                    if c in prev.columns
                ]
            )
            .sort("residual_K")
        )
        show_scrollable(prev, height=320)

,label,n,mae_K,bias_K,rmse_K,corr,within_1K
0,yesterday_2026-08-20,18,1.984,-0.132,2.375,0.36,0.278


C:\Users\ckaplinger\AppData\Local\Temp\ipykernel_25568\4022300383.py:32: DeprecationWarning: Casting from String to Date is deprecated and will be removed in Polars 2.0.
Use `str.to_date()` instead.
  .with_columns(pl.col("game_date").cast(pl.Date))


player_name,pitcher_team,away_team,home_team,line,lean,expected_K,actual_K,residual_K,projected_tbf,actual_PA
Gavin Williams,CLE,SF,CLE,7.5,under,6.46,11.0,-4.54,23.94,24.0
Ian Seymour,TB,TOR,TB,5.5,under,3.93,8.0,-4.07,22.35,22.0
Jacob deGrom,TEX,WSH,TEX,6.5,under,6.03,10.0,-3.97,22.73,20.0
Shane Bieber,TOR,TOR,TB,3.5,over,4.43,7.0,-2.57,22.34,25.0
Gerrit Cole,NYY,NYY,BAL,6.5,under,5.97,8.0,-2.03,24.35,23.0
Grayson Rodriguez,LAA,LAA,HOU,4.5,over,4.90,6.0,-1.10,22.64,27.0
Andrew Alvarez,WSH,WSH,TEX,4.5,over,4.57,5.0,-0.43,21.81,22.0
Anthony Kay,CWS,ATL,CWS,4.5,over,5.74,6.0,-0.26,23.34,24.0
Randy Dobnak,KC,ATH,KC,3.5,over,3.92,4.0,-0.08,23.58,20.0
Michael McGreevy,STL,STL,CIN,4.5,under,4.63,4.0,0.63,23.08,20.0


## 3. Today — preferred projections

`pitcher_team`, name, matchup, `xK`, and over-line probabilities.

In [ ]:
pref = _attach_pitcher_team(preferred)
p_cols = [c for c in pref.columns if c.startswith("p_over_")]
front = [
    c
    for c in ("pitcher_team", "player_name", "away_team", "home_team", "expected_K")
    if c in pref.columns
]

round_exprs = [pl.col(c).round(3) for c in p_cols if pref[c].dtype in (pl.Float32, pl.Float64)]
if "expected_K" in pref.columns:
    round_exprs.append(pl.col("expected_K").round(2))

today_view = (
    pref.select(front + p_cols)
    .sort("expected_K", descending=True)
    .with_columns(round_exprs)
)

print(len(today_view))
show_scrollable(today_view, height=480)

30


pitcher_team,player_name,away_team,home_team,expected_K,p_over_2_5,p_over_2_5_cal,p_over_3_5,p_over_3_5_cal,p_over_4_5,p_over_4_5_cal,p_over_5_5,p_over_5_5_cal,p_over_6_5,p_over_6_5_cal,p_over_7_5,p_over_7_5_cal,p_over_8_5,p_over_8_5_cal,p_over_9_5,p_over_9_5_cal
MIL,Jacob Misiorowski,ATL,MIL,7.34,0.989,0.985,0.959,0.947,0.892,0.872,0.776,0.776,0.616,0.618,0.438,0.456,0.276,0.304,0.152,0.181
ATL,Chris Sale,ATL,MIL,7.03,0.987,0.982,0.953,0.939,0.878,0.856,0.752,0.753,0.586,0.590,0.408,0.428,0.250,0.279,0.134,0.162
NYY,Cam Schlittler,TOR,NYY,6.54,0.978,0.970,0.928,0.908,0.829,0.803,0.680,0.682,0.501,0.510,0.328,0.353,0.188,0.218,0.094,0.118
CWS,Sean Burke,NYM,CWS,6.46,0.977,0.969,0.926,0.906,0.825,0.799,0.674,0.676,0.495,0.504,0.322,0.348,0.184,0.213,0.092,0.115
LAD,Yoshinobu Yamamoto,PIT,LAD,6.41,0.973,0.965,0.918,0.896,0.811,0.784,0.655,0.658,0.475,0.486,0.305,0.332,0.172,0.201,0.085,0.108
PHI,Jesus Luzardo,STL,PHI,6.40,0.972,0.963,0.913,0.890,0.802,0.774,0.641,0.644,0.458,0.470,0.289,0.317,0.160,0.188,0.077,0.099
TEX,MacKenzie Gore,LAA,TEX,6.14,0.965,0.954,0.897,0.871,0.773,0.745,0.604,0.607,0.419,0.432,0.255,0.284,0.136,0.163,0.063,0.083
SEA,Emerson Hancock,CHC,SEA,5.93,0.959,0.947,0.883,0.854,0.750,0.720,0.573,0.577,0.388,0.403,0.230,0.259,0.119,0.145,0.053,0.071
LAA,Reid Detmers,LAA,TEX,5.54,0.947,0.931,0.854,0.821,0.702,0.672,0.514,0.518,0.330,0.347,0.184,0.213,0.089,0.112,0.037,0.051
DET,Troy Melton,DET,KC,5.30,0.920,0.899,0.800,0.760,0.623,0.594,0.426,0.431,0.253,0.272,0.129,0.157,0.057,0.076,0.022,0.032


## 4. Today — edges (model × books)

Reads `artifacts/odds_log/recommendations.parquet` from `production/odds/odds_board.py`.

Important runtime context:
- Active policy mode is typically **conservative** (`edge_floor=0.12`) unless you changed CLI flags.
- This notebook applies operating-profile filters from `kpi_policy` (edge/rest/TBF/quality-gate).
- Low BET counts can be real (tight floor + segment filters + quote coverage), not necessarily a bug.

This view shows only **BET** rows by default; toggle `SHOW_ALL_EDGES=True` to inspect skipped/OOS rows.

In [ ]:
SHOW_ALL_EDGES = False  # True -> include skip / OOS
# Optional local override: set to profile name (A_edge12/B_edge14/C_over14_under12) or keep None for policy default.
OPERATING_PROFILE_OVERRIDE: str | None = None

REC_META_PATH = ROOT / "artifacts" / "odds_log" / "recommendations_meta.json"
ENSEMBLE_CFG_PATH = ROOT / "production" / "ops" / "live_krate_ensemble.json"

_policy = load_kpi_policy()
_oper = _policy.get("operating_profile", {}) if isinstance(_policy, dict) else {}
OPERATING_PROFILE = OPERATING_PROFILE_OVERRIDE or str(_oper.get("name", "A_edge12"))
REST_MAX_EXCLUSIVE = float(_oper.get("filters", {}).get("rest_max_exclusive", 45.0))
TBF_MIN = float(_oper.get("filters", {}).get("tbf_min", 15.0))
PROFILE_DEFS = _oper.get("profiles", {}) if isinstance(_oper.get("profiles", {}), dict) else {}


def apply_operating_profile(df: pl.DataFrame, profile: str) -> pl.DataFrame:
    out = df
    if "edge" not in out.columns or "best_side" not in out.columns:
        return out

    prof = PROFILE_DEFS.get(profile, {}) if isinstance(PROFILE_DEFS, dict) else {}
    edge_min = prof.get("edge_min")
    edge_min_over = prof.get("edge_min_over")
    edge_min_under = prof.get("edge_min_under")

    if edge_min is not None:
        out = out.filter(pl.col("edge") >= float(edge_min))
    elif edge_min_over is not None or edge_min_under is not None:
        over_thr = float(edge_min_over if edge_min_over is not None else 0.0)
        under_thr = float(edge_min_under if edge_min_under is not None else 0.0)
        out = out.filter(
            ((pl.col("best_side") == "over") & (pl.col("edge") >= over_thr))
            | ((pl.col("best_side") == "under") & (pl.col("edge") >= under_thr))
        )

    if "days_rest" in out.columns:
        out = out.filter(pl.col("days_rest").is_null() | (pl.col("days_rest") < REST_MAX_EXCLUSIVE))
    if "projected_tbf" in out.columns:
        out = out.filter(pl.col("projected_tbf").is_null() | (pl.col("projected_tbf") >= TBF_MIN))
    if "quality_gate_block" in out.columns:
        out = out.filter(~pl.col("quality_gate_block").fill_null(False))
    return out


if not REC_PATH.exists():
    print(f"Missing {REC_PATH}. Run: python production/odds/odds_board.py --unit 50 --roi-mode conservative")
else:
    rec = pl.read_parquet(REC_PATH)
    if "game_date" in rec.columns:
        rec = rec.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("game_date"))
        rec = rec.filter(pl.col("game_date") == slate_date[:10])

    if not SHOW_ALL_EDGES and "recommendation" in rec.columns:
        edges = rec.filter(pl.col("recommendation") == "BET")
    else:
        edges = rec

    before_profile_n = edges.height
    edges = apply_operating_profile(edges, OPERATING_PROFILE)
    after_profile_n = edges.height

    edge_cols = [
        c
        for c in (
            "recommendation",
            "pitcher_team",
            "player_name",
            "away_team",
            "home_team",
            "expected_K",
            "book",
            "line",
            "best_side",
            "best_price",
            "edge",
            "units",
            "stake",
            "days_rest",
            "projected_tbf",
        )
        if c in edges.columns
    ]
    view = edges.select(edge_cols)
    if "edge" in view.columns:
        view = view.with_columns((pl.col("edge") * 100).round(1).alias("edge_pct")).drop("edge")
    if "expected_K" in view.columns:
        view = view.with_columns(pl.col("expected_K").round(2))

    if REC_META_PATH.exists():
        import json

        meta = json.loads(REC_META_PATH.read_text(encoding="utf-8"))
        print(
            "meta:",
            f"slate={meta.get('slate_date')} raw_matched={meta.get('n_matched_raw')} filtered_matched={meta.get('n_matched')} "
            f"BET={meta.get('n_bet')} edge_floor={meta.get('edge_floor')}",
        )

    if view.is_empty():
        print("No rows survive active operating filters for this slate.")
    else:
        show_scrollable(view, height=360)

    print(
        f"profile={OPERATING_PROFILE} | recommendations n={rec.height} | pre_profile={before_profile_n} | shown={after_profile_n}"
    )

live_krate_config: manual_best_aug21_deduped_transfer
weights_note: 0.00 sparse72 + 0.60 sparse72_monotone + 0.40 final58
meta: slate=2026-08-21 raw_matched=53 filtered_matched=27 BET=3 edge_floor=0.12


recommendation,pitcher_team,player_name,away_team,home_team,expected_K,book,line,best_side,best_price,units,stake,days_rest,projected_tbf,edge_pct
BET,MIL,Jacob Misiorowski,ATL,MIL,7.34,draftkings,9.5,under,-134.0,1.72,85.92,6.0,23.390,27.9
BET,WSH,Brad Lord,WSH,MIA,3.97,fanduel,2.5,over,-106.0,1.13,56.66,12.0,21.037,26.0
BET,ATH,J.T. Ginn,ATH,HOU,4.28,draftkings,4.5,under,-102.0,0.78,38.99,6.0,23.032,14.2


profile=A_edge12 | recommendations n=27 | pre_profile=3 | shown=3


In [ ]:
# 4b. Quick diagnostics (use results_gate_policy for full threshold sweeps)
if not REC_PATH.exists():
    print(f"Missing {REC_PATH}. Run: python production/odds/odds_board.py --unit 50")
else:
    rec_raw = pl.read_parquet(REC_PATH)
    if "game_date" in rec_raw.columns:
        rec_raw = rec_raw.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("game_date"))
        rec_raw = rec_raw.filter(pl.col("game_date") == slate_date[:10])

    rec = rec_raw
    if "best_side" in rec.columns and "side" not in rec.columns:
        rec = rec.with_columns(pl.col("best_side").alias("side"))
    rec_best = keep_best_available_lines(rec)

    print(f"recommendations raw_n={rec_raw.height} best_line_n={rec_best.height}")
    print("recommendation mix")
    if "recommendation" in rec_best.columns:
        by_rec = rec_best.group_by("recommendation").agg(pl.len().alias("n")).sort("n", descending=True)
        show_table(by_rec)

    if "quality_gate_reason" in rec_best.columns:
        print("\nquality gate reasons")
        by_gate = (
            rec_best.with_columns(pl.col("quality_gate_reason").fill_null("none").alias("quality_gate_reason"))
            .group_by("quality_gate_reason")
            .agg(pl.len().alias("n"))
            .sort("n", descending=True)
        )
        show_table(by_gate)

    if "oos_reason" in rec_best.columns:
        print("\nOOS reasons")
        by_oos = (
            rec_best.with_columns(pl.col("oos_reason").fill_null("in_support").alias("oos_reason"))
            .group_by("oos_reason")
            .agg(pl.len().alias("n"))
            .sort("n", descending=True)
        )
        show_table(by_oos)

    side_health = side_clv_roi_health(rec_best)
    if side_health.height:
        print("\nside CLV/ROI health")
        show_table(side_health)
        if has_over_clv_red_flag(side_health):
            print("RED FLAG: over mean_clv_pp <= 0 (or missing).")
    elif LEDGER_PATH.exists():
        ledger_hist = pl.read_parquet(LEDGER_PATH)
        hist_settled = ledger_hist.filter(
            (pl.col("status") == "settled") & (pl.col("stake").cast(pl.Float64).fill_null(0.0) > 0)
        ) if not ledger_hist.is_empty() else ledger_hist
        if "game_date" in hist_settled.columns:
            hist_settled = hist_settled.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("game_date"))
        hist_best = keep_best_available_lines(hist_settled)
        hist_side_health = side_clv_roi_health(hist_best)
        if hist_side_health.height:
            print("\nhistorical side CLV/ROI health (best-line settled)")
            show_table(hist_side_health)
            if has_over_clv_red_flag(hist_side_health):
                print("RED FLAG: over mean_clv_pp <= 0 (or missing).")

    print("\nFor full profile and threshold selection, run production/notebooks/results_bettable_cohort.ipynb and production/notebooks/results_gate_policy.ipynb")


recommendations raw_n=27 best_line_n=27
recommendation mix


,recommendation,n
0,skip,23
1,BET,3
2,OOS,1



quality gate reasons


,quality_gate_reason,n
0,,27



OOS reasons


,oos_reason,n
0,in_support,26
1,projected_tbf<12,1



historical side CLV/ROI health (best-line settled)


,side,n,stake,pnl,roi,mean_clv_pp
0,over,88,5805.355,-841.042,-0.145,0.010
1,under,91,7195.898,891.325,0.124,0.005



For full profile and threshold selection, run production/notebooks/results_bettable_cohort.ipynb and production/notebooks/results_gate_policy.ipynb
